# Continuous position versus binary half on identical sequence attempts

This notebook isolates the effect of representing position continuously or as a first/later-half indicator. Both models use the **same eligible attempt rows**, Bernoulli-logit likelihood, fixed work-mode interaction, optimizer, and random-effects structure. Exercise Elo is deliberately excluded from both models:

`success ~ centered_work_mode * model_position`  
`        + (1 | classroom_id) + (1 | student_id)`  
`        + (1 + model_position | sequence_id)`

The continuous model uses normalized position from 0 to 1. The binary model uses `later_half = 1` only when normalized position is greater than 0.5. An exact midpoint in an odd-length sequence remains in the first half, so no attempt is removed. This binary model is intentionally not the earlier sequence-level half-half change-score model: it is an attempt-level matched model designed to make position representation the only substantive modeling difference.

## 1. Setup

In [1]:
from __future__ import annotations

import gc
import importlib
import sys
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = next(
        parent for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]
        if (parent / 'pyproject.toml').exists()
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.model_work_mode_sequence_position_comparison as comparison_model  # noqa: E402

importlib.reload(comparison_model)

from scripts.model_work_mode_progress import load_attempts, split_populations  # noqa: E402
from scripts.model_work_mode_sequence_position_comparison import (  # noqa: E402
    BINARY_HALF,
    CONTINUOUS,
    add_binary_half_indicator,
    build_adjusted_probability_curve,
    build_half_assignment_audit,
    build_position_comparison_trajectory,
    fit_position_model,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 150)
print(f'Kernel Python: {sys.executable}')

Kernel Python: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\.venv\Scripts\python.exe


## 2. Parameters

The combined population matches the earlier half-half report and includes exclusive-mode and both-mode students. `MIN_SEQUENCE_EXERCISES = 4` preserves the established eligible sequence definition.

In [2]:
INPUT_FILE = PROJECT_ROOT / 'data_MIA' / '986-neurips-mia_20260415_100024.parquet'
EXERCISE_CATALOG_JSON = PROJECT_ROOT / 'data_MIA' / 'exo_mia.json'
MODULE_CONFIG_JSON = PROJECT_ROOT / 'data_MIA' / 'config_mia.json'

MODEL_POPULATION = 'combined'
MIN_SEQUENCE_EXERCISES = 4
POSITION_BINS = 20
MAXITER = 200
RUN_MODELS = True
MODEL_ENCODINGS = (CONTINUOUS, BINARY_HALF)
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'work_mode_sequence_position_comparison_notebook'

if MODEL_POPULATION not in {'exclusive_modes', 'both_modes', 'combined'}:
    raise ValueError('MODEL_POPULATION must be exclusive_modes, both_modes, or combined')
for required_path in (INPUT_FILE, EXERCISE_CATALOG_JSON, MODULE_CONFIG_JSON):
    if not required_path.exists():
        raise FileNotFoundError(required_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = Namespace(
    input_file=INPUT_FILE,
    input_csv=None,
    data_dir=None,
    exercise_catalog_json=EXERCISE_CATALOG_JSON,
    module_config_json=MODULE_CONFIG_JSON,
    keep_only_single_module_playlists=False,
)

## 3. Build one shared eligible trajectory

Playlist activities are qualified by playlist ID, only the earliest retained student-exercise attempt is used, and sequences shorter than four unique exercises are removed. Exercise Elo is neither attached nor used to filter rows. The binary indicator is then added to this one trajectory table.

In [3]:
attempts = load_attempts(args)
if MODEL_POPULATION == 'combined':
    selected_attempts = attempts
else:
    population_frames = split_populations(attempts)
    selected_attempts = population_frames[MODEL_POPULATION]
    del population_frames

attempt_summary = pd.DataFrame([{
    'population': MODEL_POPULATION,
    'attempt_rows_before_sequence_filter': len(selected_attempts),
    'students': selected_attempts['student_id'].nunique(),
    'classrooms': selected_attempts['classroom_id'].nunique(),
    'modules': selected_attempts['module'].nunique(),
}])
display(attempt_summary)

trajectory = build_position_comparison_trajectory(
    selected_attempts,
    min_sequence_exercises=MIN_SEQUENCE_EXERCISES,
)
del attempts, selected_attempts
gc.collect()

sequence_summary = (
    trajectory.groupby('work_mode', as_index=False, observed=True)
    .agg(
        attempt_rows=('success', 'size'),
        sequences=('sequence_id', 'nunique'),
        students=('student_id', 'nunique'),
        classrooms=('classroom_id', 'nunique'),
    )
)
half_assignment_audit = build_half_assignment_audit(trajectory)
midpoint_rows = int(trajectory['is_exact_midpoint'].sum())
model_sample_audit = pd.DataFrame([
    {
        'position_encoding': encoding,
        'attempt_rows': len(trajectory),
        'sequences': trajectory['sequence_id'].nunique(),
        'students': trajectory['student_id'].nunique(),
        'classrooms': trajectory['classroom_id'].nunique(),
        'exact_midpoint_rows_retained': midpoint_rows,
    }
    for encoding in MODEL_ENCODINGS
])
assert model_sample_audit['attempt_rows'].nunique() == 1
assert model_sample_audit['sequences'].nunique() == 1
assert 'exercise_elo' not in trajectory.columns
display(sequence_summary.round(2))
display(half_assignment_audit)
display(model_sample_audit.round(3))

,population,attempt_rows_before_sequence_filter,students,classrooms,modules
0,combined,5590740,37894,3091,27


,work_mode,attempt_rows,sequences,students,classrooms
0,playlist,1306573,118104,13667,1076
1,zpdes,2472741,349909,25489,2501


,work_mode,half,attempt_rows,sequences,students,exact_midpoint_rows
0,playlist,first,679077,118104,13667,51581
1,playlist,later,627496,118104,13667,0
2,zpdes,first,1295946,349909,25489,119151
3,zpdes,later,1176795,349909,25489,0


,position_encoding,attempt_rows,sequences,students,classrooms,exact_midpoint_rows_retained
0,continuous,3779314,468013,34224,2910,170732
1,binary_half,3779314,468013,34224,2910,170732


## 4. Descriptive position profile

The same observed binned success profile is shown behind both fitted representations. The vertical line marks the binary threshold; a position exactly equal to 0.5 belongs to the first half.

In [4]:
position_bin_index = np.minimum(
    (trajectory['normalized_position'].astype(float) * POSITION_BINS).astype(int),
    POSITION_BINS - 1,
)
binned_trajectory = (
    trajectory.assign(position_bin_index=position_bin_index)
    .groupby(['work_mode', 'position_bin_index'], as_index=False, observed=True)
    .agg(
        success_rate=('success', 'mean'),
        attempt_rows=('success', 'size'),
        sequences=('sequence_id', 'nunique'),
    )
)
binned_trajectory['normalized_position'] = (
    binned_trajectory['position_bin_index'] + 0.5
) / POSITION_BINS

colors = {'playlist': '#DD8452', 'zpdes': '#4C72B0'}
descriptive_figure = go.Figure()
for work_mode in ('playlist', 'zpdes'):
    rows = binned_trajectory[binned_trajectory['work_mode'].eq(work_mode)]
    label = 'ZPDES' if work_mode == 'zpdes' else 'Playlist'
    descriptive_figure.add_trace(
        go.Scatter(
            x=rows['normalized_position'], y=rows['success_rate'],
            mode='lines+markers', name=f'{label}: observed success',
            line={'color': colors[work_mode], 'width': 3},
        )
    )
descriptive_figure.add_vline(x=0.5, line_dash='dot', line_color='gray')
descriptive_figure.update_layout(
    title='Observed success over the shared sequence positions',
    template='simple_white', hovermode='x unified',
    legend={'orientation': 'h', 'y': 1.15, 'x': 0.5, 'xanchor': 'center'},
)
descriptive_figure.update_xaxes(title_text='Normalized position')
descriptive_figure.update_yaxes(title_text='Observed success', tickformat='.0%')
descriptive_figure.show(config={'displaylogo': False, 'responsive': True})

## 5. Fit the two matched models

Each model receives the same `trajectory` table. Both use centered work mode, position centered at 0.5, classroom/student/sequence random intercepts, and an independent sequence random coefficient for the fitted position variable. Exercise Elo is absent from both fixed-effect matrices. Results are checkpointed after each fit because the full models are computationally expensive.

In [5]:
fit_results = {}
if RUN_MODELS:
    for encoding in MODEL_ENCODINGS:
        print(f'Fitting {encoding} model on {len(trajectory):,} identical attempt rows...')
        result = fit_position_model(
            trajectory,
            position_encoding=encoding,
            population=MODEL_POPULATION,
            maxiter=MAXITER,
        )
        fit_results[encoding] = result
        pd.DataFrame([result.summary]).to_csv(
            OUTPUT_DIR / f'{encoding}_model_summary.csv', index=False, encoding='utf-8-sig'
        )
        if not result.fixed_effects.empty:
            result.fixed_effects.to_csv(
                OUTPUT_DIR / f'{encoding}_fixed_effects.csv', index=False, encoding='utf-8-sig'
            )
            result.variance_components.to_csv(
                OUTPUT_DIR / f'{encoding}_variance_components.csv', index=False, encoding='utf-8-sig'
            )
            result.adjusted_changes.to_csv(
                OUTPUT_DIR / f'{encoding}_adjusted_changes.csv', index=False, encoding='utf-8-sig'
            )
        display(pd.DataFrame([result.summary]).round(4))
        gc.collect()
else:
    print('RUN_MODELS is False; the shared sample and descriptive audit are complete.')

Fitting continuous model on 3,779,314 identical attempt rows...


C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



,population,position_encoding,position_definition,trajectory_rows,n_rows,n_sequences,n_students,n_classrooms,n_modules,model_specification,status,converged,reportable,interaction_reportable,probability_levels_reportable,iterations,zpdes_reference,position_reference,playlist_log_odds_change,zpdes_log_odds_change,interaction_log_odds,interaction_std_error,interaction_ci_low,interaction_ci_high,interaction_p_value,playlist_adjusted_change_points,zpdes_adjusted_change_points,adjusted_difference_in_change_points,invalid_standard_error_terms,error
0,combined,continuous,normalized_position from 0 to 1,3779314,3779314,468013,34224,2910,27,"Bernoulli-logit: centered work mode * centered model position; random intercepts for classroom, student, and sequence; independent random continuo...",ok,True,True,True,False,18,0.6543,0.5,0.2628,1.5789,1.3162,0.0089,1.2988,1.3335,0.0,4.561,32.612,28.051,Intercept,None


Fitting binary_half model on 3,779,314 identical attempt rows...


,population,position_encoding,position_definition,trajectory_rows,n_rows,n_sequences,n_students,n_classrooms,n_modules,model_specification,status,converged,reportable,interaction_reportable,probability_levels_reportable,iterations,zpdes_reference,position_reference,playlist_log_odds_change,zpdes_log_odds_change,interaction_log_odds,interaction_std_error,interaction_ci_low,interaction_ci_high,interaction_p_value,playlist_adjusted_change_points,zpdes_adjusted_change_points,adjusted_difference_in_change_points,invalid_standard_error_terms,error
0,combined,binary_half,later_half = 1 when normalized_position > 0.5; midpoint retained in first half,3779314,3779314,468013,34224,2910,27,"Bernoulli-logit: centered work mode * centered model position; random intercepts for classroom, student, and sequence; independent random binary_h...",ok,True,True,True,False,27,0.6543,0.5,0.1246,0.9411,0.8166,0.0058,0.8053,0.8278,0.0,2.3187,20.5344,18.2157,Intercept,None


## 6. Side-by-side results

The continuous interaction is the ZPDES-minus-playlist difference in the modeled start-to-end slope. The binary interaction is the ZPDES-minus-playlist difference in the first-to-later-half log-odds change. Their magnitudes need not be identical because the binary contrast replaces all within-half position information with one step.

In [6]:
model_summary = pd.DataFrame()
teacher_summary = pd.DataFrame()
fixed_effects = pd.DataFrame()
variance_components = pd.DataFrame()
adjusted_changes = pd.DataFrame()
model_curves = pd.DataFrame()

if fit_results:
    model_summary = pd.DataFrame([result.summary for result in fit_results.values()])
    teacher_summary = model_summary[[
        'position_encoding', 'status', 'converged', 'interaction_reportable',
        'probability_levels_reportable', 'n_rows', 'n_sequences', 'n_students',
        'playlist_log_odds_change', 'zpdes_log_odds_change',
        'interaction_log_odds', 'interaction_std_error',
        'interaction_ci_low', 'interaction_ci_high', 'interaction_p_value',
        'playlist_adjusted_change_points', 'zpdes_adjusted_change_points',
        'adjusted_difference_in_change_points', 'invalid_standard_error_terms',
    ]].copy()
    fixed_effects = pd.concat(
        [result.fixed_effects for result in fit_results.values()], ignore_index=True
    )
    variance_components = pd.concat(
        [result.variance_components for result in fit_results.values()], ignore_index=True
    )
    adjusted_changes = pd.concat(
        [result.adjusted_changes for result in fit_results.values()], ignore_index=True
    )
    curve_frames = []
    for encoding, result in fit_results.items():
        if result.fixed_effects.empty:
            continue
        curve_frames.append(build_adjusted_probability_curve(
            result.fixed_effects,
            position_encoding=encoding,
            population=MODEL_POPULATION,
            zpdes_reference=float(result.summary['zpdes_reference']),
        ))
    model_curves = pd.concat(curve_frames, ignore_index=True) if curve_frames else pd.DataFrame()

    display(teacher_summary.round(4))
    probability_table = adjusted_changes.copy()
    probability_table['adjusted_period_0_percent'] = (
        probability_table['adjusted_period_0_probability'] * 100
    )
    probability_table['adjusted_period_1_percent'] = (
        probability_table['adjusted_period_1_probability'] * 100
    )
    display(probability_table[[
        'position_encoding', 'work_mode', 'period_0_label', 'period_1_label',
        'adjusted_period_0_percent', 'adjusted_period_1_percent',
        'adjusted_change_points',
    ]].round(2))
    display(fixed_effects.round(4))
    display(variance_components.round(6))

    for row in teacher_summary.itertuples(index=False):
        if not row.interaction_reportable:
            print(f'{row.position_encoding}: interaction inference is not reportable.')
        elif not row.probability_levels_reportable:
            print(
                f'{row.position_encoding}: interaction is reportable; probability changes are point estimates.'
            )

,position_encoding,status,converged,interaction_reportable,probability_levels_reportable,n_rows,n_sequences,n_students,playlist_log_odds_change,zpdes_log_odds_change,interaction_log_odds,interaction_std_error,interaction_ci_low,interaction_ci_high,interaction_p_value,playlist_adjusted_change_points,zpdes_adjusted_change_points,adjusted_difference_in_change_points,invalid_standard_error_terms
0,continuous,ok,True,True,False,3779314,468013,34224,0.2628,1.5789,1.3162,0.0089,1.2988,1.3335,0.0,4.5610,32.6120,28.0510,Intercept
1,binary_half,ok,True,True,False,3779314,468013,34224,0.1246,0.9411,0.8166,0.0058,0.8053,0.8278,0.0,2.3187,20.5344,18.2157,Intercept


,position_encoding,work_mode,period_0_label,period_1_label,adjusted_period_0_percent,adjusted_period_1_percent,adjusted_change_points
0,continuous,playlist,start,end,75.28,79.84,4.56
1,continuous,zpdes,start,end,50.67,83.28,32.61
2,binary_half,playlist,first_half,later_half,74.09,76.41,2.32
3,binary_half,zpdes,first_half,later_half,56.01,76.54,20.53


,population,position_encoding,term,estimate,std_error,z_value,p_value,ci_low,ci_high,odds_ratio
0,combined,continuous,Intercept,0.9644,NaN,NaN,NaN,NaN,NaN,2.6233
1,combined,continuous,zpdes,-0.4286,0.0097,-44.0438,0.0,-0.4477,-0.4095,0.6514
2,combined,continuous,position,1.1239,0.0041,271.8109,0.0,1.1158,1.1320,3.0769
3,combined,continuous,zpdes_x_position,1.3162,0.0089,148.4577,0.0,1.2988,1.3335,3.7291
4,combined,binary_half,Intercept,0.8505,NaN,NaN,NaN,NaN,NaN,2.3409
5,combined,binary_half,zpdes,-0.4009,0.0135,-29.7715,0.0,-0.4273,-0.3745,0.6697
6,combined,binary_half,position,0.6588,0.0027,246.2290,0.0,0.6536,0.6641,1.9325
7,combined,binary_half,zpdes_x_position,0.8166,0.0058,141.8490,0.0,0.8053,0.8278,2.2627


,population,position_encoding,group,variance
0,combined,continuous,classroom_id,0.289419
1,combined,continuous,student_id,0.393990
2,combined,continuous,sequence_id,1.237695
3,combined,continuous,sequence_id_rand_coef_position,0.260454
4,combined,binary_half,classroom_id,0.381047
5,combined,binary_half,student_id,0.412374
6,combined,binary_half,sequence_id,1.195715
7,combined,binary_half,sequence_id_rand_coef_position,0.083551


continuous: interaction is reportable; probability changes are point estimates.
binary_half: interaction is reportable; probability changes are point estimates.


In [7]:
comparison_figure = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    subplot_titles=('Continuous normalized position', 'Binary first/later half'),
)
if not model_curves.empty:
    for column, encoding in enumerate(MODEL_ENCODINGS, start=1):
        for work_mode in ('playlist', 'zpdes'):
            observed = binned_trajectory[binned_trajectory['work_mode'].eq(work_mode)]
            predicted = model_curves[
                model_curves['position_encoding'].eq(encoding)
                & model_curves['work_mode'].eq(work_mode)
            ]
            label = 'ZPDES' if work_mode == 'zpdes' else 'Playlist'
            comparison_figure.add_trace(
                go.Scatter(
                    x=observed['normalized_position'], y=observed['success_rate'],
                    mode='markers', name=f'{label}: observed',
                    marker={'color': colors[work_mode], 'opacity': 0.45},
                    legendgroup=f'{work_mode}_observed', showlegend=column == 1,
                ), row=1, col=column,
            )
            comparison_figure.add_trace(
                go.Scatter(
                    x=predicted['normalized_position'], y=predicted['adjusted_probability'],
                    mode='lines', name=f'{label}: fitted model',
                    line={
                        'color': colors[work_mode], 'width': 3,
                        'shape': 'hv' if encoding == BINARY_HALF else 'linear',
                    },
                    legendgroup=f'{work_mode}_model', showlegend=column == 1,
                ), row=1, col=column,
            )
        comparison_figure.add_vline(x=0.5, line_dash='dot', line_color='gray', row=1, col=column)
    comparison_figure.update_layout(
        title='Same attempts and random-effects structure; only position encoding changes',
        template='simple_white',
        legend={'orientation': 'h', 'y': 1.18, 'x': 0.5, 'xanchor': 'center'},
        height=520,
    )
    comparison_figure.update_xaxes(title_text='Normalized sequence position')
    comparison_figure.update_yaxes(title_text='First-attempt success', tickformat='.0%', row=1, col=1)
    comparison_figure.show(config={'displaylogo': False, 'responsive': True})

## 7. Interpretation

Both encodings support the same conclusion. In the continuous model without Elo, the ZPDES-minus-playlist trajectory interaction is 1.3162 log-odds (SE = 0.0089, 95% CI [1.2988, 1.3335], p < .001). Its model-implied changes are +4.56 percentage points for playlist and +32.61 for ZPDES, a descriptive +28.05-point contrast.

In the matched binary-half model without Elo, the ZPDES-minus-playlist interaction is 0.8166 log-odds (SE = 0.0058, 95% CI [0.8053, 0.8278], p < .001). Its model-implied first-to-later-half changes are +2.32 points for playlist and +20.53 for ZPDES, a descriptive +18.22-point contrast. The binary effect is smaller because positions within each half are deliberately treated as equivalent.

- The continuous model measures the complete fitted trajectory from the first to the last position.
- The binary model measures one modeled step between attempts at positions up to and including 0.5 and attempts after 0.5. It deliberately ignores ordering within each half.
- Because both models use every attempt, the binary result is attempt-weighted. It is not expected to reproduce the older sequence-weighted Gaussian half-half result (+2.12 playlist and +17.39 ZPDES).
- A similar direction across the two matched models shows that the conclusion does not depend on treating position as exactly linear. A substantially weaker binary interaction would indicate that dichotomization discards important within-half trajectory information.
- Exercise Elo is deliberately excluded from both progression models. Elo patterns and work-mode calibration agreement are assessed separately.
- These are observational associations among sequences that reached at least four unique first-attempt exercises; they are not randomized causal estimates.
- As in the continuous notebook, formal inference belongs to the interaction when its standard error is valid. Probability-scale changes remain point estimates if baseline-level standard errors are incomplete.

## 8. Save combined outputs

In [8]:
attempt_summary.to_csv(OUTPUT_DIR / 'attempt_summary.csv', index=False, encoding='utf-8-sig')
sequence_summary.to_csv(OUTPUT_DIR / 'sequence_summary.csv', index=False, encoding='utf-8-sig')
half_assignment_audit.to_csv(
    OUTPUT_DIR / 'half_assignment_audit.csv', index=False, encoding='utf-8-sig'
)
model_sample_audit.to_csv(
    OUTPUT_DIR / 'model_sample_audit.csv', index=False, encoding='utf-8-sig'
)
binned_trajectory.to_csv(
    OUTPUT_DIR / 'observed_binned_trajectory.csv', index=False, encoding='utf-8-sig'
)
descriptive_figure.write_html(
    OUTPUT_DIR / 'observed_position_profile.html', include_plotlyjs='cdn'
)
if not model_summary.empty:
    model_summary.to_csv(OUTPUT_DIR / 'model_summary.csv', index=False, encoding='utf-8-sig')
    teacher_summary.to_csv(OUTPUT_DIR / 'teacher_summary.csv', index=False, encoding='utf-8-sig')
    fixed_effects.to_csv(OUTPUT_DIR / 'fixed_effects.csv', index=False, encoding='utf-8-sig')
    variance_components.to_csv(
        OUTPUT_DIR / 'variance_components.csv', index=False, encoding='utf-8-sig'
    )
    adjusted_changes.to_csv(
        OUTPUT_DIR / 'adjusted_changes.csv', index=False, encoding='utf-8-sig'
    )
    model_curves.to_csv(OUTPUT_DIR / 'model_curves.csv', index=False, encoding='utf-8-sig')
    comparison_figure.write_html(
        OUTPUT_DIR / 'continuous_vs_binary_half.html', include_plotlyjs='cdn'
    )
print(f'Saved outputs to: {OUTPUT_DIR}')

Saved outputs to: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\artifacts\work_mode_sequence_position_comparison_notebook
